# Small-signal stability analysis of the Kundur two-area system

This notebook computes the eigenvalues, damping ratios, participation factors and
mode shapes of a power system linearised about an operating point, using RAMSES
through the `stepss` package. No MATLAB is involved at any stage.

The system is Kundur's two-area, four-machine benchmark (Kundur, *Power System
Stability and Control*, McGraw-Hill 1994, Example 12.6). It is the standard test
case for inter-area oscillations: two symmetric areas joined by a weak tie, each
with two 900 MVA machines. We run it twice, with and without the power system
stabilisers, because the difference between those two runs is the clearest
possible demonstration that the analysis is doing something physically real.

## What "small-signal analysis" computes here

A dynamic power system model is a set of differential-algebraic equations. The
differential equations describe machine rotors, exciters and governors; the
algebraic equations describe the network, which has no dynamics of its own.
Linearised about an operating point, that is

$$\begin{aligned}
\Delta\dot{x} &= f_x\,\Delta x + f_y\,\Delta y \\
0 &= g_x\,\Delta x + g_y\,\Delta y
\end{aligned}$$

where $x$ are the states and $y$ the algebraic variables. Eliminating $\Delta y$
using the second equation gives the **state matrix**

$$A_{sys} = f_x - f_y\,g_y^{-1}\,g_x$$

whose eigenvalues are the system modes. This elimination is a Schur complement,
and it exists only if $g_y$ is nonsingular, which is what makes the model
index-1. RAMSES does all of this internally: it assembles the unreduced
Jacobian, factorises $g_y$ once with KLU, forms $A_{sys}$, and solves the dense
eigenproblem with LAPACK.

Each eigenvalue $\lambda = \sigma \pm j\omega$ is one mode:

- **frequency** $f = |\omega| / 2\pi$ in Hz,
- **damping ratio** $\zeta = -\sigma / |\lambda|$, which is the number engineers
  actually judge a mode by. $\zeta < 0$ means the oscillation grows: the
  operating point is unstable.

## Prerequisites, please read

**This needs a RAMSES newer than 3.60.** The `EIG` disturbance used below was
added after that release. If you are on 3.60 or earlier, the disturbance is
accepted and then does nothing useful. Check with:

```python
import stepss; print(stepss.__version__)
```

The `stepss` version's leading components name the bundled RAMSES, so anything
from the first release after 3.60 onward carries the feature.

**There is no `stepss.ssa` module yet.** Analysis is driven through the ordinary
disturbance API, which is a documented, stable interface. A dedicated Python
module with an object-oriented result type is planned; when it lands, the
file parsing in this notebook becomes unnecessary. The C entry point `run_ssa`
also exists for callers who want to trigger the analysis without a disturbance,
but it has no Python wrapper yet, so this notebook does not use it.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

import stepss

print("stepss version:", stepss.__version__)

# Everything below writes into the notebook's own directory. Keep runs of the
# two variants apart, because the results files are named from the basename you
# pass to EIG and would otherwise overwrite each other.
os.makedirs("run_pss", exist_ok=True)
os.makedirs("run_nopss", exist_ok=True)

## Step 1: describe the case

A `cfg` object lists the input files. Three points that are easy to get wrong:

- **`solveroptions.dat` must set `$OMEGA_REF SYN`.** Under the default centre-of-inertia
  reference frame the engine refuses the analysis, because the COI equations are
  computed by finite differences at export time and never enter the assembled
  Jacobian, so reducing under COI would silently hold COI speed constant and
  give you a plausible, wrong answer. The engine refuses rather than doing that.
  The bundled `solveroptions.dat` already sets `SYN`.
- **`$SCHEME DE` is required too.** Under the integrated scheme the pure
  differential-algebraic values exist only briefly inside the Newton loop, so
  the analysis refuses there as well.
- **A disturbance file is mandatory even though we inject the disturbance from
  Python.** `nothing.dst` exists only to satisfy that requirement.

In [ ]:
def build_case(dynfile, trj):
    """Assemble a Kundur case. `dynfile` selects the PSS variant."""
    case = stepss.cfg()
    case.addData("lf.dat")            # power flow: buses, lines, transformers, operating point
    case.addData(dynfile)             # dynamic data: machines, exc_kundur AVR/PSS, governors
    case.addData("solveroptions.dat")  # solver settings, including $OMEGA_REF SYN and $SCHEME DE
    case.addDst("nothing.dst")        # required, but contains no events
    case.addObs("obs.dat")
    case.addTrj(trj)
    return case


# The two variants differ in exactly one parameter: the PSS gain KSTAB, which is
# 20.0 in dyn.dat and 0.0 in dyn_noPSS.dat, on all four exciters. Everything else
# is identical, so any difference in the results is attributable to the PSS alone.
print(open("dyn.dat").readlines()[9].strip())
print(open("dyn_noPSS.dat").readlines()[9].strip())

## Step 2: run to the operating point and linearise there

`execSim(case, 0.0)` initialises the system and pauses at $t = 0$, the
pre-disturbance steady state. That is the operating point we linearise about.
Small-signal results are only meaningful *at* an operating point, so where you
pause determines what you get: pausing mid-swing linearises about a
non-equilibrium and the numbers describe that instant, not the steady state.

`addDisturb(t, "EIG 'basename'")` schedules the analysis. When the run reaches
`t`, RAMSES assembles the Jacobian, reduces it and writes three files named from
`basename`. We then advance just past `t` so the event actually fires.

In [ ]:
def run_ssa(dynfile, workdir, basename):
    """Run one variant and leave its three results files in `workdir`."""
    here = os.getcwd()
    os.chdir(workdir)
    try:
        for f in ("lf.dat", dynfile, "solveroptions.dat", "nothing.dst", "obs.dat"):
            if not os.path.exists(f):
                os.symlink(os.path.join(here, f), f)

        case = build_case(dynfile, "out.trj")
        ram = stepss.sim()
        ram.execSim(case, 0.0)          # initialise and pause at the operating point
        ram.addDisturb(0.001, "EIG '%s'" % basename)  # schedule the analysis
        ram.contSim(0.01)               # advance past it so the event fires
        ram.endSim()
    finally:
        os.chdir(here)


run_ssa("dyn.dat", "run_pss", "ssa")
run_ssa("dyn_noPSS.dat", "run_nopss", "ssa")

print(sorted(f for f in os.listdir("run_pss") if f.startswith("ssa")))

## Step 3: read the modes

`<basename>_modes.dat` has one line per mode. The columns are:

| column | meaning |
|---|---|
| `index` | mode number, 1-based |
| `re`, `im` | real and imaginary parts of $\lambda$ |
| `zeta` | damping ratio $-\mathrm{Re}(\lambda)/|\lambda|$ |
| `freq_hz` | $|\mathrm{Im}(\lambda)|/2\pi$ |
| `dom` | 1 if $\mathrm{Re}(\lambda)$ exceeded the `real_limit` filter |
| `smp` | **1 if the eigenvalue is simple, 0 if degenerate** |

**The `smp` column deserves attention, because ignoring it produces
confident nonsense.** Power system spectra are heavily degenerate: identical
machine models with identical parameters produce identical poles, and this
4-machine system has 20 of its 70 modes sharing an eigenvalue with another mode.
In a degenerate eigenspace the individual eigenvectors are *not unique*, so the
participation factors and mode shape of such a mode are basis-dependent. They
are real numbers that mean nothing physically and would come out differently on
another LAPACK build. Treat `smp == 0` rows as unusable for participation
analysis. The header records the gap tolerance used to decide.

In [ ]:
def read_modes(path):
    """Return a structured array of modes. Blank file sections are impossible:
    every mode is always written, filtered or not."""
    raw = np.loadtxt(path, comments="#")
    return {
        "index": raw[:, 0].astype(int),
        "lam":   raw[:, 1] + 1j * raw[:, 2],
        "zeta":  raw[:, 3],
        "freq":  raw[:, 4],
        "dom":   raw[:, 5].astype(bool),
        "simple": raw[:, 6].astype(bool),
    }


modes_pss = read_modes("run_pss/ssa_modes.dat")
modes_nopss = read_modes("run_nopss/ssa_modes.dat")

for tag, m in (("with PSS", modes_pss), ("without PSS", modes_nopss)):
    print("%-12s %3d modes, %3d simple, %3d degenerate"
          % (tag, len(m["index"]), m["simple"].sum(), (~m["simple"]).sum()))

## Step 4: find the electromechanical modes

Rotor oscillations in a system like this sit roughly between 0.1 and 2.5 Hz.
Below that are slow controller modes, above it exciter and network dynamics.
We take one member of each conjugate pair, by keeping only $\mathrm{Im}(\lambda) > 0$.

In [ ]:
def electromechanical(m, lo=0.1, hi=2.5):
    sel = (m["freq"] > lo) & (m["freq"] < hi) & (m["lam"].imag > 0)
    order = np.argsort(m["freq"][sel])
    return {k: v[sel][order] for k, v in m.items()}


for tag, m in (("WITH PSS", modes_pss), ("WITHOUT PSS", modes_nopss)):
    em = electromechanical(m)
    print("\n%s" % tag)
    print("  %-5s %-9s %-10s %-28s %s" % ("mode", "f [Hz]", "zeta", "lambda", "simple"))
    for i in range(len(em["index"])):
        print("  %-5d %-9.4f %-+10.4f %-28s %s"
              % (em["index"][i], em["freq"][i], em["zeta"][i],
                 "%+.4f %+.4fj" % (em["lam"][i].real, em["lam"][i].imag),
                 "yes" if em["simple"][i] else "NO"))

## Step 5: the result that matters

Look at the mode near **0.62 Hz** in both tables.

- Without the PSS its damping ratio is about **-0.023**: negative, so the
  oscillation grows and the operating point is small-signal unstable.
- With the PSS it is about **+0.109**: comfortably damped.

That is the inter-area mode, in which the two areas swing against each other,
and it is the mode the stabilisers exist to damp. The sign flip is the whole
point of the exercise, and it reproduces Kundur's Example 12.6.

The two local modes near 1.1 Hz are the machines within each area swinging
against each other. They are damped in both cases, and the PSS pushes them
higher in frequency and much better damped.

In [ ]:
def interarea(m, lo=0.4, hi=0.9):
    """The single mode in the inter-area band, taking one of each conjugate pair."""
    sel = (m["freq"] > lo) & (m["freq"] < hi) & (m["lam"].imag > 0)
    assert sel.sum() == 1, "expected exactly one inter-area mode, found %d" % sel.sum()
    return {k: v[sel][0] for k, v in m.items()}


ia_pss, ia_nopss = interarea(modes_pss), interarea(modes_nopss)

print("inter-area mode")
print("  without PSS: f = %.4f Hz, zeta = %+.4f  -> %s"
      % (ia_nopss["freq"], ia_nopss["zeta"],
         "UNSTABLE" if ia_nopss["zeta"] < 0 else "stable"))
print("  with    PSS: f = %.4f Hz, zeta = %+.4f  -> %s"
      % (ia_pss["freq"], ia_pss["zeta"],
         "UNSTABLE" if ia_pss["zeta"] < 0 else "stable"))

## Step 6: participation factors, or which machines are involved

An eigenvalue tells you a mode exists. It does not tell you *where* it lives.
The participation factor of state $k$ in mode $i$,

$$p_{ki} = |w_{ki}\, v_{ki}|$$

built from the left and right eigenvectors, measures how strongly that state
takes part. RAMSES normalises each mode's column so its largest entry is 1, which
removes the arbitrary scaling of the eigenvectors.

`<basename>_pf.dat` carries these, one row per (mode, state) pair above a
threshold, with the family, device and variable name inlined so you do not have
to cross-reference anything. Rows are written only for modes that passed the
`real_limit` filter.

Looking at the `omega` states, one per machine rotor, tells you which machines
swing in each mode.

In [ ]:
def read_pf(path):
    rows = []
    with open(path) as fh:
        for line in fh:
            if line.startswith("#") or not line.strip():
                continue
            f = line.split()
            rows.append((int(f[0]), int(f[1]), float(f[2]), f[3], f[4], f[5]))
    return rows


def omega_participation(path, mode_index):
    """Participation of each machine's rotor speed in one mode."""
    return {dev: pf for m, _s, pf, _fam, dev, var in read_pf(path)
            if m == mode_index and var == "omega"}


print("omega participation, without PSS\n")
for label, idx in (("inter-area %.3f Hz" % ia_nopss["freq"], ia_nopss["index"]),):
    print("  %s:" % label)
    for dev, pf in sorted(omega_participation("run_nopss/ssa_pf.dat", idx).items()):
        print("    %-6s %.3f" % (dev, pf))

em_nopss = electromechanical(modes_nopss)
for i in range(len(em_nopss["index"])):
    if em_nopss["freq"][i] < 0.9:
        continue
    print("\n  local mode %.3f Hz:" % em_nopss["freq"][i])
    for dev, pf in sorted(omega_participation("run_nopss/ssa_pf.dat",
                                              em_nopss["index"][i]).items()):
        print("    %-6s %.3f" % (dev, pf))

Read that output carefully, because it is the physical interpretation the whole
analysis exists to support:

- the **inter-area** mode lists all four machines, with substantial
  participation from each;
- the **1.085 Hz** mode lists only G1 and G2, so it is area 1's local mode;
- the **1.116 Hz** mode lists only G3 and G4, the mirror image in area 2.

**Note what the absence means.** G3 and G4 do not appear under the 1.085 Hz mode
because their participation fell below the `pf_threshold` the analysis was run
with, not because it is exactly zero. It is around 0.02 to 0.04 against roughly
1.0 for the area-1 machines, so two orders of magnitude separate the
participating machines from the rest. That is a decisive separation, not a
marginal one. Lower the threshold if you want the small entries written out;
the value used is recorded in the `_modes.dat` header.

If your own model produces a mode you cannot attribute, participation factors
are the tool that tells you which device to go and look at.

## Step 7: mode shapes, or how the machines move relative to each other

`<basename>_ms.dat` gives the rotor-speed component of each machine in a mode,
as magnitude and angle. Magnitudes are normalised so the largest is 1, and
**angles are relative to that largest entry**, because an eigenvector's absolute
phase is arbitrary and would otherwise vary run to run.

For an inter-area mode the expected picture is two groups roughly 180 degrees
apart: the areas swinging against each other.

In [ ]:
def read_ms(path, mode_index):
    out = []
    with open(path) as fh:
        for line in fh:
            if line.startswith("#") or not line.strip():
                continue
            f = line.split()
            if int(f[0]) == mode_index:
                out.append((f[4], float(f[2]), float(f[3])))
    return out


shape = read_ms("run_nopss/ssa_ms.dat", ia_nopss["index"])
print("inter-area mode shape, without PSS\n")
for dev, mag, ang in shape:
    print("  %-6s magnitude %.3f   angle %+8.2f deg" % (dev, mag, ang))

fig, ax = plt.subplots(figsize=(5, 5), subplot_kw={"projection": "polar"})
for dev, mag, ang in shape:
    th = np.deg2rad(ang)
    ax.annotate("", xy=(th, mag), xytext=(0, 0),
                arrowprops=dict(arrowstyle="->", lw=2))
    ax.text(th, mag * 1.12, dev, ha="center", va="center")
ax.set_rmax(1.3)
ax.set_title("Inter-area mode shape (rotor speeds), no PSS", pad=18)
plt.show()

The two areas sit roughly opposite each other on the dial. That is what
"inter-area" means, made visible.

## Step 8: the s-plane

The conventional summary plot. The vertical axis is oscillation frequency, the
horizontal axis is decay rate. **Everything strictly left of the imaginary axis
is stable**; anything on or right of it is not. The dashed rays mark constant
damping ratio, the usual planning criterion being $\zeta \geq 0.05$.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)

for ax, (tag, m) in zip(axes, (("without PSS", modes_nopss), ("with PSS", modes_pss))):
    em = electromechanical(m)
    ax.axvline(0.0, color="crimson", lw=1.5, zorder=1)        # the stability boundary
    for z in (0.05, 0.10):                                     # constant-damping rays
        ax.plot([0, -3], [0, 3 * np.sqrt(1 - z**2) / z], "--",
                color="0.6", lw=1, zorder=1)
    ax.scatter(em["lam"].real, em["lam"].imag, s=90,
               facecolors="none", edgecolors="tab:blue", zorder=3)
    for i in range(len(em["index"])):
        ax.annotate(" %.2f Hz" % em["freq"][i],
                    (em["lam"][i].real, em["lam"][i].imag), fontsize=8)
    unstable = em["zeta"] < 0
    if unstable.any():
        ax.scatter(em["lam"].real[unstable], em["lam"].imag[unstable],
                   s=160, marker="x", color="crimson", zorder=4,
                   label="unstable")
        ax.legend(loc="lower left")
    ax.set_title("Electromechanical modes, %s" % tag)
    ax.set_xlabel(r"Re$(\lambda)$  [1/s]")
    ax.grid(alpha=0.3)

axes[0].set_ylabel(r"Im$(\lambda)$  [rad/s]")
axes[0].set_xlim(-3.0, 0.5)
axes[0].set_ylim(0, 9)
plt.tight_layout()
plt.show()

Without the PSS one mode sits to the *right* of the red line. That single point
is the whole result: the operating point is small-signal unstable, and a
disturbance would excite a growing 0.62 Hz oscillation between the two areas.

## Where to go next

- Change the PSS gain `KSTAB` in `dyn.dat` and watch the inter-area mode move.
- Stress the tie line in `lf.dat` by raising the transfer, and watch damping fall.
- Analyse at several operating points by re-running the whole flow per point.
  **Finalise each run with `endSim()` before loading the next case**: a paused
  simulation is not finished, and loading a new case without finalising silently
  resumes the old one.
- For systems above a few thousand states the engine refuses the dense solve, by
  design, and reports the `$EIG_MAX_STATES` limit. That regime needs sparse
  shift-invert methods, which `scipy.sparse.linalg.eigs` can drive from the
  Jacobian returned by `getJac()`.